In [1]:
import numpy as np
import cv2
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os
import re
import glob

In [14]:
class Single_Image_Wall:
    def __init__(self, YOLOimage_path, RAWimage_folder_path, output_path, filename, cut_interval_proportion, thick_th_proportion, min_spacing_proportion):
        """   
        Parameters:
        YOLOimage_path: str, path of specific image processed in YOLO model detection
        RAWimage_folder_path: str, path of folder containing raw images, each image prefiex should find match with prefiex of image used in YOLO         output_path: str, file path for saving walls detected in imahge
        output_path: str, file path for saving detected wall as image
        filename: str, name of image processed in YOLO model detection
        cut_interval_proportion: float (0-1), determine size of pieces cutting form image (cut interval = img_size*cut_interval_proportion)
        thick_th_proportion: float (0-1), determine minimum thickness of wall detected (thickness threshold = img_size*thick_th_proportion)
        min_spacing_proportion: float (0-1), determine minimum space between walls detected (minimum space = img_size*min_spacing_proportion)
        """
        self.cut_interval_proportion = cut_interval_proportion
        self.thick_th_proportion = thick_th_proportion
        self.min_spacing_proportion = min_spacing_proportion
        self.YOLOimage_name = filename
        self.YOLOimage_path = YOLOimage_path
        self.RAWimage_path = self.find_RAWimage_path(RAWimage_folder_path)
        self.preprocessed_image = self.preprocess_image()
        self.wall_detected = self.get_wall_result()
        self.image_result_path =  output_path.replace(".png", "_Wall.png") 

    def find_RAWimage_path(self, RAWimage_folder_path):
        def get_prefix(filename):
            match = re.match(r'^([A-Za-z0-9]+)', filename)  
            return match.group(1) if match else None

        base = get_prefix(self.YOLOimage_name)
        matches = glob.glob(os.path.join(RAWimage_folder_path, base + '*'))
        if matches:
            return matches[0]
        else:
            print(f"image is not found with prefix {base}")
            return None
    
    def preprocess_image(self, threshold=100):
        image = cv2.imread(self.RAWimage_path, cv2.IMREAD_GRAYSCALE)
        self.original_image_shape = image.shape[:2] 
        image = cv2.resize(image, (512, 512), interpolation=cv2.INTER_LANCZOS4)
        filtered_img = np.copy(image)
        if filtered_img.dtype != np.uint8:
            filtered_img = filtered_img.astype(np.uint8)

        filtered_img[filtered_img > threshold] = 255 #only keep black (edges)
        filtered_img[filtered_img <= threshold] = 0 
        return filtered_img

    def find_vertical_wall_pieces(self, sub_img, thickness_th, min_spacing):
        """   
        Parameters:
          sub_img: image piece that used for detection
          thickness_th: least width in pixels of the bar to detect
          min_spacing: minimum gap in columns between separate bars (to avoid duplicates)
        Returns:
          thick_wall: black walls that can use with specific thickness threshold
        """
        H, W = sub_img.shape

        # Create binary mask where black pixels are 1
        black_mask = (sub_img == 0).astype(np.uint8)

        # Row-wise cumulative sum across columns for fast window sums
        cumsum = np.pad(np.cumsum(black_mask, axis=1), ((0,0),(1,0)), mode='constant', constant_values=0)
        """
        [[1, 0, 1]    to     [[0 1 1 2]
        [0, 1, 1]]           [0 0 1 2]]
        """
        detected = []
        for start in range(0, W - thickness_th + 1):
            end = start + thickness_th
            #check if line equal to thickness, give 1d array that each number is thickness of each row
            window_row_counts = cumsum[:, end] - cumsum[:, start]  
            if np.all(window_row_counts == thickness_th): 
                detected.append((start, end))
        
        #if no line: return empty result
        if not detected:
            thick_wall = np.full_like(sub_img, 255, dtype=np.uint8)
            return thick_wall

        # Merge detections that are overlapping or closer than min_spacing
        merged = []
        current_s, current_e = detected[0]
        for s, e in detected[1:]:
            if s <= current_e + min_spacing:
                current_e = max(current_e, e)
            else:
                merged.append((current_s, current_e))
                current_s, current_e = s, e
        merged.append((current_s, current_e))

        # Build output array: background 255, detected bar pixels 0
        thick_wall = np.copy(sub_img)
        thick_wall[thick_wall == 0] = 255
        for s, e in merged:
            thick_wall[:, int(s):int(e)] = 0
        
        return thick_wall

    def find_horizontal_wall_pieces(self, sub_img, thickness_th, min_spacing):
        """   
        Parameters:
          sub_img: image piece that used for detection
          thickness_th: least width in pixels of the bar to detect
          min_spacing: minimum gap in columns between separate bars (to avoid duplicates)
        Returns:
          thick_wall: black walls that can use with specific thickness threshold
        """
        H, W = sub_img.shape

        # Create binary mask where black pixels are 1
        black_mask = (sub_img == 0).astype(np.uint8)

        # Row-wise cumulative sum across columns for fast window sums
        cumsum = np.pad(np.cumsum(black_mask, axis=0), ((1,0),(0,0)), mode='constant', constant_values=0)
        """
                          [[0 0],
       [[1, 0],    to     [1, 0],
       [1, 1],            [2, 1],
       [0, 1]]            [2, 2]]
        """
        detected = []
        for start in range(0, H - thickness_th + 1):
            end = start + thickness_th
            #check if line equal to thickness
            window_col_counts = cumsum[end,:] - cumsum[start,:]  
            if np.all(window_col_counts == thickness_th): 
                detected.append((start, end))
        # if no line: return empty result
        if not detected:
            thick_wall = np.full_like(sub_img, 255, dtype=np.uint8)
            return thick_wall

        # Merge detections that are overlapping or closer than min_spacing
        merged = []
        current_s, current_e = detected[0]
        for s, e in detected[1:]:
            if s <= current_e + min_spacing:
                current_e = max(current_e, e)
            else:
                merged.append((current_s, current_e))
                current_s, current_e = s, e
        merged.append((current_s, current_e))

        # Build output array: background 255, detected bar pixels 0
        thick_wall = np.copy(sub_img)
        thick_wall[thick_wall == 0] = 255
        for s, e in merged:
            thick_wall[int(s):int(e),:] = 0

        return thick_wall

    def detect_thick_wall_single(self, img, orient='v'): 
        """
        input:
          orient: orientation of wall ('h'= horizontal, 'v'=vertical)
        """
        img_size = 512
        
        interval = int(img_size*self.cut_interval_proportion) #cut image into pieces
        thick_th = int(img_size*self.thick_th_proportion)
        min_spac = int(img_size*self.min_spacing_proportion)
        window_start = 0
        window_end = interval
        detected_wall = []

        while window_start < img_size:
            if window_end > img_size:
                window_end = img_size

            if (orient=='v'):
                sub_img = img[window_start:window_end,:]
                detected_piece = self.find_vertical_wall_pieces(sub_img, 
                                                       thickness_th=thick_th,
                                                       min_spacing=min_spac)
            elif(orient=='h'): #problem
                sub_img = img[:, window_start:window_end]
                detected_piece = self.find_horizontal_wall_pieces(sub_img, 
                                                         thickness_th=thick_th, 
                                                         min_spacing=min_spac)
            else:
                print("a valid orientation need to be provided")
                return detected_wall
            
            detected_wall.append(detected_piece)
            window_start = window_start + interval
            window_end = window_end + interval
            
        return detected_wall

    def label_wall(self,croped_img):
        vertical_wall = self.detect_thick_wall_single(croped_img, orient='v')
        horizontal_wall = self.detect_thick_wall_single(croped_img, orient='h')
        vertical_wall = np.vstack(vertical_wall)
        horizontal_wall = np.hstack(horizontal_wall)
        
        all_thick_wall = vertical_wall * horizontal_wall
        all_thick_wall = all_thick_wall*255
        return all_thick_wall, vertical_wall, horizontal_wall 

    def rotation_detect(self, angle=0):
        orig_h, orig_w = self.preprocessed_image.shape
        center = (orig_w / 2, orig_h / 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        cos = abs(M[0, 0])
        sin = abs(M[0, 1])
        new_w = int((orig_h * sin) + (orig_w * cos))
        new_h = int((orig_h * cos) + (orig_w * sin))
        M[0, 2] += (new_w / 2) - center[0]
        M[1, 2] += (new_h / 2) - center[1]
        rotated = cv2.warpAffine(self.preprocessed_image, M, (new_w, new_h),
                                 flags=cv2.INTER_NEAREST, #keep output only have value 0,255
                                 borderMode=cv2.BORDER_CONSTANT, 
                                 borderValue=255)

        n_center = int(new_w / 2)
        length = int(orig_w / 2)
        crop_img = rotated[n_center-length:n_center+length, n_center-length:n_center+length]
        
        #detection
        wall_result, _, _= self.label_wall(crop_img)

        #resize back
        expanded = np.full((new_h, new_w), 255, dtype=np.uint8)   
        top = (new_h - orig_h) // 2
        left = (new_w - orig_w) // 2
        expanded[top:top+orig_h, left:left+orig_w] = wall_result
        R = M[:2, :2]        
        t = M[:2, 2]
        R_inv = np.linalg.inv(R)
        t_inv = -R_inv @ t
        M_inv = np.hstack((R_inv, t_inv.reshape(-1, 1)))
        final_output = cv2.warpAffine(expanded, M_inv, (orig_w, orig_h),
                                      flags=cv2.INTER_NEAREST, #keep output only have value 0,255
                                      borderMode=cv2.BORDER_CONSTANT,
                                      borderValue=255)
        return final_output

    def get_wall_result(self):
        thick_wall = np.copy(self.preprocessed_image)
        thick_wall[thick_wall == 0] = 255
        for i in range(0, 91):
            final = self.rotation_detect(angle=i)
            thick_wall = thick_wall*final #255*0=0 (white to black), 0*0=0 (keep black even overlap)
            thick_wall = thick_wall*255 #255*255=1 (white to white in uint8) so expand to 255
        thick_wall = cv2.resize(thick_wall, (self.original_image_shape[1], self.original_image_shape[0]), 
                              interpolation=cv2.INTER_NEAREST)
        thick_wall = cv2.cvtColor(thick_wall, cv2.COLOR_GRAY2RGB)
        return thick_wall

    def save_wall_result(self):
        cv2.imwrite(self.image_result_path, self.wall_detected)

In [15]:
class Floor_Wall_Folder_Processor:
    def __init__(self, YOLOimage_folder_path, RAWimage_folder_path, output_folder_path, cut_interval_proportion, thick_th_proportion, min_spacing_proportion):
        """
        Parameters:
        YOLOimage_folder_path: str, path of images processed in YOLO model detection
        RAWimage_folder_path: str, path of folder containing raw images, each image prefiex should find match with prefiex of image used in YOLO 
        output_folder_path: str, folder path for saving detected walls as images
        cut_interval_proportion: float (0-1), determine size of pieces cutting form image (cut interval = img_size*cut_interval_proportion)
        thick_th_proportion: float (0-1), determine minimum thickness of wall detected (thickness threshold = img_size*thick_th_proportion)
        min_spacing_proportion: float (0-1), determine minimum space between walls detected (minimum space = img_size*min_spacing_proportion)
        """
        self.cut_interval_proportion = cut_interval_proportion
        self.thick_th_proportion = thick_th_proportion
        self.min_spacing_proportion = min_spacing_proportion
        self.YOLOimage_folder_path = YOLOimage_folder_path
        self.RAWimage_folder_path = RAWimage_folder_path
        self.output_folder_path = output_folder_path

    def Load_folder(self):
        for filename in os.listdir(self.YOLOimage_folder_path):
            YOLOimage_file_path = os.path.join(self.YOLOimage_folder_path, filename)
            output_file_path = os.path.join(self.output_folder_path, filename)
            wall = Single_Image_Wall(YOLOimage_file_path, 
                                     RAWimage_folder_path, 
                                     output_file_path, 
                                     filename,
                                     cut_interval_proportion = self.cut_interval_proportion,
                                     thick_th_proportion = self.thick_th_proportion,
                                     min_spacing_proportion = self.min_spacing_proportion)
            wall.save_wall_result()   
            print(f"{filename} Processed")

In [ ]:
YOLOimage_folder_path = ". . /Floorplan/dataset/images/test" # Add path of test image
RAWimage_folder_path = ". . /Floorplan/dataset/original_images/test" # Add path of original image
output_folder_path = ". . /Floorplan/text_result" #Add output path
cut_interval_proportion = 0.02
thick_th_proportion = 0.01 
min_spacing_proportion = 0.01 

FloorPlan = Floor_Wall_Folder_Processor(YOLOimage_folder_path, 
                                        RAWimage_folder_path, 
                                        output_folder_path, 
                                        cut_interval_proportion, 
                                        thick_th_proportion, 
                                        min_spacing_proportion)

FloorPlan.Load_folder()